# Treinamento da BearingCNN no Dataset Paderborn (PU Dataset)

**Projeto:** Diagnóstico de Falhas em Rolamentos Industriais via CWT + CNN  
**Dataset:** Paderborn University (PU Dataset — 64 kHz, Falhas Reais por Fadiga)  
**Classes:** 3 Classes (`normal`, `inner_race`, `outer_race`)  
**Objetivo:** Treinar do zero a arquitetura convolucional própria `BearingCNN` (com *Global Average Pooling*) sobre escalogramas CWT gerados a partir dos sinais de alta resolução de Paderborn.

---

## 1. Importação de Módulos e Configurações

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

# Fixar seeds para 100% de reprodutibilidade
from src.config import RANDOM_SEED, PADERBORN_PROCESSED_DIR, NUM_EPOCHS, LEARNING_RATE
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

from src.cnn_processor import train_and_evaluate_bearing_cnn, get_data_loaders
from src.visualization import plot_training_curves, plot_confusion_matrix

## 2. Treinamento da BearingCNN (Própria) com Checkpointing Automático

In [ ]:
# Treinamento da rede própria sobre o dataset Paderborn processado
res_cnn = train_and_evaluate_bearing_cnn(
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    data_dir=PADERBORN_PROCESSED_DIR,
    checkpoint_name="checkpoint_paderborn_bearing_cnn_best.pth"
)

## 3. Avaliação e Gráficos de Desempenho

In [ ]:
# 1. Curvas de Aprendizado (Loss e Acurácia)
plot_training_curves(
    res_cnn['history'],
    model_name="BearingCNN (Paderborn)",
    save_path="../docs/images/training_curves_paderborn_bearing_cnn.png"
)

# 2. Relatório de Classificação no Teste
print("=" * 80)
print(" RELATÓRIO DE CLASSIFICAÇÃO NO TESTE (PADERBORN — BEARING CNN) ")
print("=" * 80)
print(classification_report(
    res_cnn['test_labels'],
    res_cnn['test_preds'],
    target_names=res_cnn['classes'],
    digits=4
))

# 3. Matriz de Confusão no Teste Inédito
plot_confusion_matrix(
    res_cnn,
    model_name="BearingCNN — Paderborn",
    save_path="../docs/images/confusion_matrix_paderborn_bearing_cnn.png"
)